# Intelligent Early-Stage Diabetes Risk Stratification for Rural Health Posts in Nepal: A Comparative Evaluation of Machine Learning Models

## Import Dataset and Generate EDA

In [ ]:
# import libraries
import pandas as pd
import matplotlib.pyplot as plt 
import seaborn as sns
import numpy as np
pd.set_option('display.max_columns', None)

In [ ]:
# load dataset
df = pd.read_csv("./datasets/final.csv")
df.head()

### Count of targets

In [ ]:
plt.figure(figsize=(7, 5))
ax = sns.countplot(data=df, x='diabetes_target', palette="Set2")
plt.title('Target Class Distribution (0: Healthy, 1: Diabetic/Prediabetic)', fontsize=12, fontweight='bold')
plt.xlabel('Diabetes Status', fontsize=10)
plt.ylabel('Patient Count', fontsize=10)
plt.xticks([0, 1], ['Non-Diabetic (0)', 'Diabetic / Prediabetic (1)'])

# annotate counts on top of bars
for p in ax.patches:
    ax.annotate(f'{int(p.get_height()):,}', (p.get_x() + p.get_width() / 2., p.get_height()),
                ha='center', va='center', xytext=(0, 5), textcoords='offset points', fontsize=10)

plt.tight_layout()
plt.savefig("./resources/target_class_distribution.png")
plt.show()
plt.close()

### Co-relataion heatmap

In [ ]:
plt.figure(figsize=(14, 10))
# computes correlation matrix of df
corr = df.corr()

sns.heatmap(
    corr,
    cmap='coolwarm', 
    annot=True, 
    fmt=".2f", 
    linewidths=0.5, 
    )
plt.title('Correlation Heatmap of Health Indicators', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig("./resources/correlation_heatmap_of_health_indicators.png")
plt.show()
plt.close()

### BMI Distribution BoxPlot

In [ ]:
plt.figure(figsize=(8, 5))

sns.kdeplot(
    data=df[df["diabetes_target"] == 0],
    x="bmi",
    fill=True,
    alpha=0.5,
    label="Non-Diabetic"
)

sns.kdeplot(
    data=df[df["diabetes_target"] == 1],
    x="bmi",
    fill=True,
    alpha=0.5,
    label="Diabetic / Prediabetic"
)

plt.title("BMI Density by Diabetes Status", fontsize=12, fontweight="bold")
plt.xlabel("Body Mass Index (BMI)")
plt.ylabel("Density")
plt.legend()

plt.tight_layout()
plt.savefig("./resources/bmi_density_plot.png")
plt.show()
plt.close()

### Split and Scale
Scaling is required, so the value comes between same scale range. Split for making train, val and test sets.

In [ ]:
# import libraries
from sklearn.preprocessing import StandardScaler

In [ ]:
# Split data to train, test, val
train_len = int(len(df) * 0.65)
val_len = train_len + int(len(df) * 0.2)

train = df.iloc[:train_len]
val = df.iloc[train_len:val_len]
test = df.iloc[val_len:]

print(f"Train Set shape = {train.shape}")
print(f"Validation Set shape = {val.shape}")
print(f"Test Set shape = {test.shape}")


In [ ]:
# continuous value
continuous = [
    "bmi",
    "poor_mental_health_days",
    "poor_physical_health_days",
]
# scaler 
scaler = {}
# continuous looping
for col in continuous:
    scaler[col] = StandardScaler()
    train[col] = scaler[col].fit_transform(train[[col]]).flatten()
    val[col] = scaler[col].transform(val[[col]]).flatten()
    test[col] = scaler[col].transform(test[[col]]).flatten()


## Predict Diabetic or Not Using Decision Tree and Evaluate

In [ ]:
# import library
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, f1_score, recall_score, precision_score, roc_curve, auc

In [ ]:
# use decision tree 
dt = DecisionTreeClassifier(
    criterion = "gini",
)

In [ ]:
dt.fit(train.drop(columns=["diabetes_target"]), train["diabetes_target"])

In [ ]:
train_pred = dt.predict(train.drop(columns=["diabetes_target"]))
train_target = train["diabetes_target"].to_list()

# calculate all metrics
